In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

base = expfolder( "system" )[2]

runsim = false;

In [ ]:
include( "size-0-params.jl" )

# Select density range.
println( "Density: ", ρlist )

# Select speed range.
s = slist[Ns]
println( "Speed:   ", s )

# System parameters.
N = 128;  N̂ = 1024

# Extract critical noise.
folderlist = [findfolder( N, ρlist[i], s; base=base, crit=true ) for i ∈ 1:Nρ]
f̂olderlist = [findfolder( N̂, ρlist[i], s; base=base, crit=true ) for i ∈ 1:Nρ]

# Unpack the kernel parameters.
f0 = 100.0
γlist = first.( readdlm.( folderlist.*"kernel-parameters-1.txt" ) );

In [ ]:
# Import area and perimeter scaling values.
βlist = first.( readdlm.( folderlist.*"area-perimeter-scaling.txt" ), 2 )

# Import the cluster change frequency values.
θlist = first.( transpose.( readdlm.( folderlist.*"optimal-rates-hat.txt" ) ), 3 );

In [ ]:
# Unpack the probability map data.
ℓlist = [polyloglaw.( 1:N̂; β=βlist[i] ) for i ∈ 1:Nρ]
Alist = [(ℓlist[i].^2)./(4π) for i ∈ 1:Nρ];

In [ ]:
# Import cluster mass distribution extracted from data.
F1list = vec.( readdlm.( folderlist.*"cluster-frequency.txt" ) )
F2list = vec.( readdlm.( f̂olderlist.*"cluster-frequency.txt" ) )

# Normalize distributions.
# f1list = [(1:N).*F1list[i]./sum( (1:N).*F1list[i] ) for i ∈ 1:Nρ]
# f2list = [(1:N̂).*F2list[i]./sum( (1:N̂).*F2list[i] ) for i ∈ 1:Nρ]
f1list = [F1list[i]./sum( F1list[i] ) for i ∈ 1:Nρ]
f2list = [F2list[i]./sum( F2list[i] ) for i ∈ 1:Nρ];

In [ ]:
# Import kernel coefficient parameters.
αlist = vec.( readdlm.( folderlist.*"kernel-decay-strength.txt" ) )

# Adjust the coagulation-fragmentation ratio for the kernel strength.
γ̂list = f0.*(2.0.^first.( αlist ).*N̂.^last.( αlist ));

In [ ]:
# Simulate the master equations in the larger system size..
T = 10.0
f̂1list = vec.( readdlm.( folderlist.*"cluster-distribution-hat.txt" ) );
f̂2list = vec.( readdlm.( f̂olderlist.*"cluster-distribution-hat.txt" ) );

In [ ]:
# Extract the power-law measured from the simulation.
λ1list = first.( readdlm.( folderlist.*"power-law.txt" ) )
λ2list = first.( readdlm.( f̂olderlist.*"power-law.txt" ) )

# Compute the power-law for the CMD from the master equations.
λ̂1list = [10^optimize( createzipfloss( N, N.*f̂1list[i] ), [log10( 1.5 )] ).minimizer[1] for i ∈ 1:Nρ]
λ̂2list = [10^optimize( createzipfloss( N̂, N̂.*f̂2list[i] ), [log10( 1.5 )] ).minimizer[1] for i ∈ 1:Nρ];

In [ ]:
# Compute the percent difference in 1 across the density speed range.
folderdata = [findfolder( N, ρlist[i], slist[j]; base=base, crit=true ) for i ∈ 1:Nρ, j ∈ 1:Ns]
λ1data = first.( readdlm.( folderdata.*"power-law.txt" ) );

# Compute the power-law over the calibrated range.
λ̂1data = first.( readdlm.( folderdata.*"power-law-hat.txt" ) )

# Compute the percent difference.
δλdata = abs.( λ1data .- λ̂1data )./λ1data
δλmax = 0.01ceil( maximum( δλdata )/0.01 );

In [ ]:
i = Nρ-1
println( "Density: ", ρlist[i] )

# Plot bounds.
ιlist = 0.0 .< F2list[i]
xmax = ceil( log10( N̂ ) )
ymin = floor( minimum( log10.( f2list[i][ιlist] ) ) )

# Initialize plot variable.
plt = plot(; size=(250,275), legend=true )

# Plot the final distribution from the mean-field equations against the measured.
plot!( plt; legend=:bottomleft, right_margin=0pt )

scatter!( plt, f2list[i][ιlist]; color=:gray, markerstrokewidth=0, label=" raw data" )
scatter!( plt, f̂2list[i][ιlist]; color=:black, markersize=3, markerstrokewidth=0, lw=2, label=" master eq." )

xticks = round.( defInt, 2.0.^(0:2:10) )
plot!( plt; xlims=extrema( xticks ), xscale=:log10, xticks=(xticks,string.(xticks)) )
plot!( plt; ylims=10.0.^[ymin,0], yscale=:log10 )

plot!( plt; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt; title="(b)", titleposition=:left )

plot!( plt; framestyle=:box )

plot!( plt; left_margin=0pt, bottom_margin=-5pt, right_margin=10pt )

# saveplot( plt, figurefolder*"master-equations-cmd_N-$(N)-$(N̂)"; background=:white, dpi=600 );

In [ ]:
i = Nρ-1
println( "Density: ", ρlist[i] )

# Plot bounds.
ιlist = 0.0 .< F2list[i]
xmax = ceil( log10( N̂ ) )
ymin = floor( minimum( log10.( f2list[i][ιlist] ) ) )

# Initialize plot variable.
plt = plot(; size=(250,275), legend=true )

# Plot the final distribution from the mean-field equations against the measured.
plot!( plt; legend=:bottomleft, right_margin=0pt )

plot!( plt, ρlist, λ2list; color=:black, lw=2 )
plot!( plt, ρlist, λ̂2list; color=:cornflowerblue, lw=2 )

plot!( plt; xlims=extrema( ρlist ), xscale=:log10 )
plot!( plt; ylims=(1,3) )

plot!( plt; xlabel="density, "*L"ρ", ylabel="power-law exponent, "*L"λ" )
plot!( plt; title="(b)", titleposition=:left )

plot!( plt; framestyle=:box )

plot!( plt; left_margin=0pt, bottom_margin=-5pt, right_margin=10pt )

# saveplot( plt, figurefolder*"master-equations-cmd_N-$(N)-$(N̂)"; background=:white, dpi=600 );

In [ ]:
# Compute the percent error between the steady state master equations and measured.
ςdata = hcat( [abs.( f2list[i] .- f̂2list[i] )./f2list[i] for i ∈ 1:Nρ]... )
ιdata = hcat( [0.0 .< f2list[i] for i ∈ 1:Nρ]... )
ςdata[.!ιdata] .= NaN;

In [ ]:
ilist = 1:Nρ

# Initialize plot variable.
plt = plot(; size=(400,275), legend=false )

for i ∈ ilist
    color = colormap1[(i-1)/Nρ]
    ιlist = 0.0 .< F2list[i]
    flist = f2list[i][ιlist]
    f̂list = f̂2list[i][ιlist]
    scatter!( plt, abs.(flist .- f̂list)./f̂list; color=color, markerstrokewidth=0 )
end

xticks = round.( defInt, 2.0.^(0:2:10) )
plot!( plt; xlims=extrema( xticks ), xscale=:log10, xticks=(xticks,string.(xticks)) )
plot!( plt; yscale=:log10 )

plot!( plt; xlabel="cluster size, "*L"i", ylabel="percentage difference,\n"*L"|P(i) - \hat P(i)|/P(i)" )
plot!( plt; title="(b)", titleposition=:left )

plot!( plt; framestyle=:box )

plot!( plt; left_margin=0pt, bottom_margin=-5pt, right_margin=10pt )

# saveplot( plt, figurefolder*"master-equations-cmd_N-$(N)-$(N̂)"; background=:white, dpi=600 );

In [ ]:
ilist = 1:Nρ

# Initialize plot variable.
plt = plot(; size=(400,275), legend=false )

heatmap!( plt, ρlist, 1:N̂, log10.( ςdata ); color=cgrad( colormap1; rev=true ), colorbar=true )

yticks = round.( defInt, 2.0.^(0:2:10) )
plot!( plt; xlims=extrema( slist ), xscale=:log10 )
plot!( plt; ylims=extrema( yticks ), yscale=:log10, yticks=(yticks,string.(yticks)) )

plot!( plt; xlabel="speed, "*L"s", ylabel="cluster size, "*L"i" )

# saveplot( plt, figurefolder*"master-equations-cmd_N-$(N)-$(N̂)"; background=:white, dpi=600 );

In [ ]:
# System sizes.
Nlist = round.( defInt, 2.0.^(3.0:7.0) )
NN = length( Nlist )

# Generate folder list for changing system size.
folderdatalist = [findfolder( Nlist[k], ρlist[i], slist[j]; base=base, crit=true )
    for k ∈ 1:NN, i ∈ 1:Nρ, j ∈ 1:Ns]

# Extract the coagulation-fragmentation rate for changing size.
γdatalist = first.( readdlm.( folderdatalist.*"kernel-parameters-1.txt" ) )

# Extract the kernel scaling coefficients.
αdata = vec.( readdlm.( folderdatalist[NN,:,:].*"kernel-decay-strength.txt" ) );

In [ ]:
i = 2
println( "Density: ", ρlist[i] )

# Plot bounds.
ιlist = 0.0 .< F1list[i]
xmax = ceil( log10( N ) )
ymin = floor( minimum( log10.( f1list[i][ιlist] ) ) )

# Initialize plot variable.
plt = plot(; layout=@layout( [a;d;e] ), size=(375,625), legend=false )

# Plot the final distribution from the mean-field equations against the measured.
scatter!( plt[1], f1list[i][ιlist]; color=:gray, markerstrokewidth=0, label=" raw data" )
scatter!( plt[1], f̂1list[i][ιlist]; color=:cornflowerblue, markersize=3, markerstrokewidth=0, lw=2, label=" c-f equations" )

xticks = round.( defInt, 2.0.^[0.0, 2.0, 5.0, 8.0, 10.0] )
yticks = 10.0.^[-7.0, -5.0, -2.0, 0.0]
plot!( plt[1]; xlims=extrema( xticks ), xscale=:log10, xticks=(xticks,string.(xticks)) )
plot!( plt[1]; ylims=extrema( yticks ), yscale=:log10, yticks=yticks )

plot!( plt[1]; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt[1]; title="(a)", titleposition=:left )

# Plot the heatmap as an inset.
pltλ = plot!( plt; inset=(1, bbox( 0.74, 0.510, 0.20, 0.40, :bottom, :left )) )

heatmap!( plt[4], ρlist, slist, δλdata; color=colormap1, colorbar=false )

plot!( plt[4]; xlims=extrema( ρlist ), xscale=:log10 )
plot!( plt[4]; ylims=extrema( slist ), yscale=:log10 )

plot!( plt[4]; xtickfontsize=8, ytickfontsize=8 )
annotate!( plt[4], 10^(-0.5), 10^(-1.40), text( L"ρ", "Computer Modern", 8, :center ) )
annotate!( plt[4], 10^(-1.40), 10^(-0.5), text( L"s", "Computer Modern", 8, :center; rotation=90 ) )

# Plot the colorbar.
pltλ̂ = plot!( plt; inset=(1, bbox( 0.74, 0.2, 0.20, 0.05, :bottom, :left )) )

xmap = 0:0.001:δλmax
ymap = [0,1]
zmap = hcat( xmap, xmap )
heatmap!( plt[5], xmap, ymap, zmap'; color=colormap1, colorbar=false )

plot!( plt[5]; xlims=(0,δλmax+eps()), xticks=[0,δλmax] )
plot!( plt[5]; yticks=[] )

plot!( plt[5]; xtickfontsize=8, ytickfontsize=8 )
annotate!( plt[5], δλmax/2, -5.0, text( "pd", "Computer Modern", 8, :center ) )

# Plot the change in the coagulation-fragmentation ratio with system size.
j = Ns  # Choose focal speed.
plot!( plt[2]; legend=false )

Nlims = vcat( extrema( Nlist )... )
for i ∈ 1:3:Nρ-1
    # Unpack power-law.
    α = αdata[i,j]
    γ̂ = (2.0^α[1]).*Nlist.^α[2]

    # Plot the data.
    color = colormap1[(i-1)/(Nρ-1)]
    scatter!( plt[2], Nlist, γdatalist[:,i,j]./f0; color=color )
    plot!( plt[2], Nlist, γ̂; color=color, lw=2 )
end

xticks = Nlist
xmin = Nlist[1] - 8
xmax = Nlist[end] + 8
plot!( plt[2]; xlims=(xmin,xmax), xticks=(xticks, string.( xticks )) )

ymin = floor( log10( minimum( γdatalist[:,:,j]./f0 ) ) )
ymax = ceil( log10( maximum( γdatalist[:,:,j]./f0 ) ) )
plot!( plt[2]; ylims=10.0.^(ymin,ymax), yscale=:log10 )

plot!( plt[2]; xlabel="system size, "*L"N", ylabel="coagulation-\nfragmentation ratio, "*L"γ" )
plot!( plt[2]; title="(b)", titleposition=:left )

# Plot the scaling parameter with changing speed and density.
pltα = plot!( plt; inset=(2, bbox( 0.74, 0.510, 0.20, 0.40, :bottom, :left )) )

heatmap!( plt[6], ρlist, slist, -last.( αdata ); color=colormap2, colorbar=false )

plot!( plt[6]; xlims=extrema( ρlist ), xscale=:log10 )
plot!( plt[6]; ylims=extrema( slist ), yscale=:log10 )
plot!( plt[6]; xtickfontsize=8, ytickfontsize=8 )

annotate!( plt[6], 10^(-0.5), 10^(-1.40), text( L"ρ", "Computer Modern", 8, :center ) )
annotate!( plt[6], 10^(-1.40), 10^(-0.5), text( L"s", "Computer Modern", 8, :center; rotation=90 ) )

# Plot the colorbar.
pltα̂ = plot!( plt; inset=(2, bbox( 0.55, 0.510, 0.025, 0.40, :bottom, :left )) )

xmap = [0,1]
ymap = 0.7:0.01:2.0
zmap = hcat( ymap, ymap )
heatmap!( plt[7], xmap, ymap, zmap; color=colormap2, colorbar=false )

plot!( plt[7]; xticks=[] )
plot!( plt[7]; yticks=ymap[[1,end]] )

plot!( plt[7]; xtickfontsize=8, ytickfontsize=8 )
annotate!( plt[7], -8.0, mean( extrema( ymap ) ), text( L"α", "Computer Modern", 8, :center; rotation=90 ) )

# Plot the extrapolated distribution.
plot!( plt[3]; legend=:bottomleft )

ιlist = 0.0 .< f2list[i]
scatter!( plt[3], f2list[i][ιlist]; color=:gray, markerstrokewidth=0, label=" raw data" )
scatter!( plt[3], f̂2list[i][ιlist]; color=:cornflowerblue, markersize=3, markerstrokewidth=0, lw=2, label=" c-f equations" )

xticks = round.( defInt, 2.0.^[0.0, 3.0, 7.0, 10.0] )
ymin = log10( minimum( f2list[i][ιlist] ) ) - 1
yticks = 10.0.^(-10:5:0)
plot!( plt[3]; xlims=extrema( xticks ), xscale=:log10, xticks=(xticks,string.(xticks)) )
plot!( plt[3]; ylims=10.0.^[-10,0], yscale=:log10, yticks=yticks )

plot!( plt[3]; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt[3]; title="(c)", titleposition=:left )

plot!( plt; left_margin=10pt, right_margin=10pt, bottom_margin=-5pt )
plot!( plt; framestyle=:box )

# saveplot( plt, figurefolder*"master-cmd-extrap_N-$(N)-$(N̂)"; background=:white, dpi=600 );

In [ ]:
i = 2
println( "Density: ", ρlist[i] )

# Plot bounds.
ιlist = 0.0 .< F1list[i]
xmax = ceil( log10( N ) )
ymin = floor( minimum( log10.( f1list[i][ιlist] ) ) )

# Initialize plot variable.
plt = plot(; layout=(2,1), size=(375,400), legend=false )

# Plot the change in the coagulation-fragmentation ratio with system size.
j = Ns  # Choose focal speed.
plot!( plt[1]; legend=false )

Nlims = vcat( extrema( Nlist )... )
for i ∈ 1:3:Nρ-1
    # Unpack power-law.
    α = αdata[i,j]
    γ̂ = (2.0^α[1]).*Nlist.^α[2]

    # Plot the data.
    color = colormap1[(i-1)/(Nρ-1)]
    scatter!( plt[1], Nlist, γdatalist[:,i,j]./f0; color=color )
    plot!( plt[1], Nlist, γ̂; color=color, lw=2 )
end

xticks = Nlist
xmin = Nlist[1] - 8
xmax = Nlist[end] + 8
plot!( plt[1]; xlims=(xmin,xmax), xticks=(xticks, string.( xticks )) )

ymin = floor( log10( minimum( γdatalist[:,:,j]./f0 ) ) )
ymax = ceil( log10( maximum( γdatalist[:,:,j]./f0 ) ) )
plot!( plt[1]; ylims=10.0.^(ymin,ymax), yscale=:log10 )

plot!( plt[1]; xlabel="system size, "*L"N", ylabel="coagulation-\nfragmentation ratio, "*L"γ" )
plot!( plt[1]; title="(a)", titleposition=:left )

# Plot the scaling parameter with changing speed and density.
pltα = plot!( plt; inset=(1, bbox( 0.75, 0.510, 0.175, 0.40, :bottom, :left )) )

heatmap!( plt[3], ρlist, slist, -last.( αdata ); color=colormap2, colorbar=false )

plot!( plt[3]; xlims=extrema( ρlist ), xscale=:log10 )
plot!( plt[3]; ylims=extrema( slist ), yscale=:log10 )
plot!( plt[3]; xtickfontsize=8, ytickfontsize=8 )

annotate!( plt[3], 10^(-0.5), 10^(-1.40), text( L"ρ", "Computer Modern", 8, :center ) )
annotate!( plt[3], 10^(-1.40), 10^(-0.5), text( L"s", "Computer Modern", 8, :center; rotation=90 ) )

# Plot the colorbar.
pltα̂ = plot!( plt; inset=(1, bbox( 0.60, 0.510, 0.025, 0.40, :bottom, :left )) )

xmap = [0,1]
ymap = 0.7:0.01:2.0
zmap = hcat( ymap, ymap )
heatmap!( plt[4], xmap, ymap, zmap; color=colormap2, colorbar=false )

plot!( plt[4]; xticks=[] )
plot!( plt[4]; yticks=ymap[[1,end]] )

plot!( plt[4]; xtickfontsize=8, ytickfontsize=8 )
annotate!( plt[4], -8.0, mean( extrema( ymap ) ), text( L"α", "Computer Modern", 8, :center; rotation=90 ) )

# Plot the extrapolated distribution.
plot!( plt[2]; legend=:bottomleft )

ιlist = 0.0 .< f2list[i]
scatter!( plt[2], f2list[i][ιlist]; color=:gray, markerstrokewidth=0, label=" raw data" )
scatter!( plt[2], f̂2list[i][ιlist]; color=:cornflowerblue, markersize=3, markerstrokewidth=0, lw=2, label=" c-f equations" )

xticks = round.( defInt, 2.0.^[0, 4, 9, 13] )
ymin = log10( minimum( f2list[i][ιlist] ) ) - 1
yticks = 10.0.^(-10:5:0)
plot!( plt[2]; xlims=2.0.^[0,13], xscale=:log10, xticks=(xticks,string.(xticks)) )
plot!( plt[2]; ylims=10.0.^[-10,0], yscale=:log10, yticks=yticks )

plot!( plt[2]; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt[2]; title="(b)", titleposition=:left )

# Plot the heatmap as an inset.
pltλ = plot!( plt; inset=(2, bbox( 0.75, 0.510, 0.175, 0.40, :bottom, :left )) )

heatmap!( plt[5], ρlist, slist, δλdata; color=colormap1, colorbar=false )

plot!( plt[5]; xlims=extrema( ρlist ), xscale=:log10 )
plot!( plt[5]; ylims=extrema( slist ), yscale=:log10 )

plot!( plt[5]; xtickfontsize=8, ytickfontsize=8 )
annotate!( plt[5], 10^(-0.5), 10^(-1.40), text( L"ρ", "Computer Modern", 8, :center ) )
annotate!( plt[5], 10^(-1.40), 10^(-0.5), text( L"s", "Computer Modern", 8, :center; rotation=90 ) )

# Plot the colorbar.
pltλ̂ = plot!( plt; inset=(2, bbox( 0.75, 0.2, 0.175, 0.05, :bottom, :left )) )

xmap = 0:0.001:δλmax
ymap = [0,1]
zmap = hcat( xmap, xmap )
heatmap!( plt[6], xmap, ymap, zmap'; color=colormap1, colorbar=false )

plot!( plt[6]; xlims=(0,δλmax+eps()), xticks=[0,δλmax] )
plot!( plt[6]; yticks=[] )

plot!( plt[6]; xtickfontsize=8, ytickfontsize=8 )
annotate!( plt[6], δλmax/2, -5.0, text( "pd", "Computer Modern", 8, :center ) )

plot!( plt; left_margin=-5pt, right_margin=10pt, bottom_margin=-5pt )
plot!( plt; framestyle=:box )

# saveplot( plt, figurefolder*"master-cmd-extrap_N-$(N)-$(N̂)"; background=:white, dpi=600 );